# PDF question answering: exploratory data analysis

Goal: find reliable evidence in a PDF and avoid unsupported answers. We inspect the frozen development corpus before comparing supervised relevance models. This notebook does not change the running app or evaluate a new final test.

In [1]:
from pathlib import Path
import sys, json
root = Path.cwd() if (Path.cwd() / "app.py").exists() else Path.cwd().parent
if not (root / "app.py").exists():
    raise RuntimeError("Run from basic-chat-app or its notebooks folder")
sys.path.insert(0, str(root))
from src.eda import write_report, render_html, NotebookHTML
summary = write_report(root / "reports")
print(json.dumps({k:v for k,v in summary.items() if k not in {"chunk_word_counts", "development_feature_correlations"}}, indent=2))

{
  "documents": 1,
  "chunks": 33,
  "questions": 41,
  "answerable": 35,
  "unanswerable": 6,
  "empty_chunk_texts": 0,
  "duplicate_questions": 0,
  "duplicate_chunk_texts": 0,
  "cases_with_unmatched_evidence": [],
  "chunk_words": {
    "min": 9,
    "median": 144.0,
    "max": 648
  },
  "chunks_per_start_page": {
    "1": 5,
    "2": 3,
    "3": 2,
    "4": 4,
    "5": 3,
    "6": 2,
    "7": 5,
    "8": 4,
    "9": 1,
    "10": 2,
    "12": 1,
    "13": 1
  },
  "development_pair_labels": {
    "known_evidence_match": 28,
    "weak_negative": 1028
  },
  "feature_names": [
    "query_coverage",
    "jaccard",
    "length_ratio",
    "log_passage_words",
    "known_evidence_match"
  ]
}


## Distributions, missing data, and correlations

Exact duplicate questions and chunks are counted. Every known evidence label is checked against the frozen chunks. Charts show length, page coverage, answerability, and the weak pair labels. Correlations use development questions only.

In [2]:
NotebookHTML(render_html(summary))

Feature,query_coverage,jaccard,length_ratio,log_passage_words,known_evidence_match
query_coverage,1.000,0.542,-0.425,0.485,0.207
jaccard,0.542,1.000,0.176,-0.227,0.115
length_ratio,-0.425,0.176,1.000,-0.870,-0.075
log_passage_words,0.485,-0.227,-0.870,1.000,0.098
known_evidence_match,0.207,0.115,-0.075,0.098,1.000


## Engineering decisions

- Preserve the existing parser, embedding normalization, thresholds, and reranker.
- Keep pages and source text with every chunk for evidence matching and citations.
- Do not randomly split passage rows: one question would leak across splits. Split questions first.
- Scale the four lexical features inside each training fold.
- Treat unmatched passages as weak negatives, not verified irrelevant passages.
- Exclude the old nine-question holdout from training and validation. It is already reported, so a final test needs new documents and labels.

In [3]:
path = root / "reports/relevance-training.json"
if path.exists():
    training = json.loads(path.read_text())
    print(json.dumps({"baseline": training["baseline_query_coverage"], "models": {name: item["validation"] for name,item in training["models"].items()}, "live_retrieval_changed": training["live_retrieval_changed"]}, indent=2))
else:
    print("Run train_relevance.py first.")

{
  "baseline": {
    "average_precision": 0.16607796414344034,
    "precision_at_0.5": 0.2,
    "recall_at_0.5": 0.14285714285714285,
    "f1_at_0.5": 0.16666666666666666,
    "answerable_question_hit_at_5": 0.42857142857142855
  },
  "models": {
    "logistic_regression": {
      "average_precision": 0.1547066599563571,
      "precision_at_0.5": 0.07246376811594203,
      "recall_at_0.5": 0.7142857142857143,
      "f1_at_0.5": 0.13157894736842105,
      "answerable_question_hit_at_5": 0.5714285714285714
    },
    "random_forest": {
      "average_precision": 0.15079365079365079,
      "precision_at_0.5": 0.07352941176470588,
      "recall_at_0.5": 0.7142857142857143,
      "f1_at_0.5": 0.13333333333333333,
      "answerable_question_hit_at_5": 0.5714285714285714
    }
  },
  "live_retrieval_changed": false
}


## Interpretation and next experiment

The supervised models are prototypes. Compare average precision against query coverage; top-five hit rate alone can hide weak precision. A classifier must not replace the current retriever just because it was trained. Retain the live pipeline until end-to-end evaluation on a larger reviewed dataset supports a change.

For a defensible final evaluation: collect several new PDFs, manually review answerable and unanswerable questions, group by document, tune on validation documents, then evaluate a locked test set once. Existing benchmark replays are regression checks, not fresh test results.